# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [ ]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [ ]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [ ]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [ ]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [ ]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [ ]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

### До питання 9. Більше епох навчання (8 замість 3)

In [ ]:
import pandas as pd
set_seed(42)
model_long = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
args_long = TrainingArguments(
    output_dir="bert-mrpc-long", num_train_epochs=8, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    eval_strategy="epoch", save_strategy="no", logging_strategy="epoch",
    fp16=torch.cuda.is_available(), report_to="none", seed=42,
)
trainer_long = Trainer(model=model_long, args=args_long, train_dataset=tokenized_datasets["train"],
                       eval_dataset=tokenized_datasets["validation"], data_collator=data_collator,
                       compute_metrics=compute_metrics)
trainer_long.train()
hist = trainer_long.state.log_history
train_loss = {round(h["epoch"]): h["loss"] for h in hist if "loss" in h}
rows = [{"епоха": round(h["epoch"]), "train loss": train_loss.get(round(h["epoch"])),
         "val loss": h["eval_loss"], "val accuracy": h["eval_accuracy"], "val F1": h["eval_f1"]}
        for h in hist if "eval_loss" in h]
pd.DataFrame(rows).round(4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.528724,0.377195,0.835784,0.880995
2,0.306423,0.353929,0.850490,0.896435
3,0.146688,0.524629,0.855392,0.890943
4,0.070833,0.643009,0.862745,0.902778
5,0.042392,0.713369,0.860294,0.902564
6,0.015914,0.816343,0.862745,0.904437
7,0.016553,0.805333,0.865196,0.905983
8,0.009529,0.784189,0.862745,0.903114


,епоха,train loss,val loss,val accuracy,val F1
0,1,0.5287,0.3772,0.8358,0.8810
1,2,0.3064,0.3539,0.8505,0.8964
2,3,0.1467,0.5246,0.8554,0.8909
3,4,0.0708,0.6430,0.8627,0.9028
4,5,0.0424,0.7134,0.8603,0.9026
5,6,0.0159,0.8163,0.8627,0.9044
6,7,0.0166,0.8053,0.8652,0.9060
7,8,0.0095,0.7842,0.8627,0.9031


## 4. Перевірка на власних прикладах

In [ ]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



### До питання 10. Ще 4 перевірки на власних прикладах

In [ ]:
my_pairs = [
    ("He was late because the train was delayed.", "The train delay made him late."),
    ("The meeting starts at 10 am.", "The meeting ends at 10 am."),
    ("I love this movie.", "I do not like this movie at all."),
    ("The bank approved my loan application.", "My loan request was accepted by the bank."),
]
for s1, s2 in my_pairs:
    print(f"{s1}\n{s2}\n→ 3 епохи: {predict(model, s1, s2)} | 8 епох: {predict(model_long, s1, s2)}\n")

He was late because the train was delayed.
The train delay made him late.
→ 3 епохи: парафраз: 98.2% | 8 епох: парафраз: 100.0%

The meeting starts at 10 am.
The meeting ends at 10 am.
→ 3 епохи: парафраз: 98.6% | 8 епох: парафраз: 100.0%

I love this movie.
I do not like this movie at all.
→ 3 епохи: парафраз: 39.1% | 8 епох: парафраз: 0.2%

The bank approved my loan application.
My loan request was accepted by the bank.
→ 3 епохи: парафраз: 98.4% | 8 епох: парафраз: 100.0%



---
# Розділ 0.1. Експерименти для відповідей на питання

Відповіді наприкінці ноутбука

### До питання 3. Пряме використання bert-base-uncased (fill-mask)

In [ ]:
for s in ["Paris is the capital of [MASK].",
          "The cat sat on the [MASK].",
          "I went to the [MASK] to buy some bread.",
          "Kyiv is the capital of [MASK]."]:
    top = unmasker(s)[:3]
    print(f"{s:45} → " + ", ".join(f"{t['token_str']} ({t['score']:.2f})" for t in top))

Paris is the capital of [MASK].               → france (0.95), algeria (0.01), morocco (0.00)
The cat sat on the [MASK].                    → floor (0.31), bed (0.12), couch (0.11)
I went to the [MASK] to buy some bread.       → market (0.78), store (0.06), bakery (0.05)
Kyiv is the capital of [MASK].                → ukraine (1.00), ukrainian (0.00), russia (0.00)


### До питання 4. Гендерні упередження в передбаченнях

In [ ]:
for subj in ["man", "woman"]:
    for tmpl in ["The {} worked as a [MASK].", "The {} was very [MASK].", "The {} is good at [MASK]."]:
        s = tmpl.format(subj)
        print(f"{s:35} → " + ", ".join(t["token_str"] for t in unmasker(s)[:5]))
    print()

The man worked as a [MASK].         → carpenter, waiter, barber, mechanic, salesman
The man was very [MASK].            → handsome, old, young, beautiful, good
The man is good at [MASK].          → it, that, this, everything, lying



[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


The woman worked as a [MASK].       → nurse, waitress, maid, prostitute, cook
The woman was very [MASK].          → pretty, beautiful, attractive, young, nice
The woman is good at [MASK].        → it, that, this, everything, lying



# 0.1. Відповіді на питання 1–10

### 1. Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?

Ноутбук виконувався в Google Colab на GPU Tesla T4. Час клітинок: імпорт бібліотек - 49 с (перше імпортування transformers/datasets), завантаження датасету MRPC — 12.5 с, завантаження `bert-base-uncased` для fill-mask — 21 с, токенізація — 4 с, донавчання на 3 епохи — 70 с (690 кроків, 1 хв 06 с тренування + оцінка), мій додатковий експеримент на 8 епох — 160 с. Разом ≈ 5.5 хв.

Проблеми із запуском:
* Першого разу Colab видав середовище без GPU (лише CPU) — клітинка 2 надрукувала «Пристрій: cpu», і тренування тривало б дуже довго; причина — тимчасово вичерпана безкоштовна квота GPU акаунта. Довелося зупинити, перемкнути тип середовища на T4 і почекати, поки GPU знову стане доступним.
* Під час переходу до наступного ноутбука Colab показав «Забагато активних
сеансів» — довелося завершити старі сеанси.
* Попередження `You are sending unauthenticated requests to the HF Hub` — не помилка, лише нижчі ліміти швидкості завантаження без HF_TOKEN. Також `LOAD REPORT` з UNEXPECTED/MISSING ключами — це очікувана поведінка (див. розділ 2 ноутбука), а не помилка.

### 2. Як Ви розумієте, що таке токен [CLS]? Яким чином він виконує свою функцію?

Токен [CLS] (classification) — спеціальний токен, який токенізатор BERT ставить на самий початок кожної послідовності. Власного змісту він не має, але проходить через усі 12 шарів трансформера разом з іншими токенами, і завдяки механізму самоуваги (self-attention) на кожному шарі збирає інформацію з усіх токенів обох речень. Тому його вихідний вектор (768 чисел) після останнього шару стає стислим представленням усієї послідовності. Під час попереднього тренування BERT цей вектор використовувався для задачі Next Sentence Prediction (чи йде речення Б за реченням А), тому модель уже вчилася класти в нього інформацію про звʼязок двох речень. Під час донавчання нова голова-класифікатор (`classifier`, лінійний шар 768 → 2) бере саме вектор [CLS] (після шару pooler) і перетворює його на два числа — оцінки для класів «не парафраз» і «парафраз».

### 3. Яке пряме застосування базової моделі bert-base-uncased?

За карткою моделі `bert-base-uncased` попередньо тренована двома задачами: Masked Language Modeling** (передбачити закрите `[MASK]` слово за контекстом з обох боків) і Next Sentence Prediction (чи йде друге речення за першим). Тому напряму, без донавчання, її можна використовувати тільки для цих задач, для заповнення пропусків через `pipeline('fill-mask')`, а також як екстрактор ознак (вектори токенів/[CLS] для пошуку схожих текстів чи кластеризації). Для генерації тексту вона не призначена — це енкодер, а не генеративна модель; для класифікації, NER чи QA її треба донавчати (як у цьому ноутбуці).

Мої перевірки fill-mask: «Paris is the capital of [MASK].» → france (0.95); «Kyiv is the capital of [MASK].» → ukraine (1.00); «I went to the [MASK] to buy some bread.» → market (0.78), store, bakery; «The cat sat on the [MASK].» → floor (0.31), bed, couch. Модель добре знає факти й типові контексти англійської мови. Модель *uncased* — весь текст переводиться в нижній регістр, тому «Paris» і «paris» для неї однакові (і відповіді виходять з малої літери).

### 4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?

Так, протидіяти варто, бо модель відтворює стереотипи з навчальних даних: для «The man worked as a [MASK]» вона пропонує carpenter, waiter, barber, mechanic, а для «The woman worked as a [MASK]» — nurse, waitress, maid і навіть prostitute. Якщо таку модель використати, наприклад, для фільтрації резюме чи генерації текстів, вона буде систематично дискримінувати людей за статтю. Способи протидії: (1) збалансувати дані — прибрати або доповнити упереджені приклади, використати контрфактичну аугментацію (для кожного речення додати копію з заміною he↔she, man↔woman); (2) донавчання на спеціально підібраних даних або з методами вирівнювання (RLHF/DPO), де модель штрафують за стереотипні відповіді; (3) пост-обробка й фільтри на виході; (4) перевірка моделей на бенчмарках упереджень (наприклад, StereoSet, CrowS-Pairs, WinoBias) перед впровадженням і документування обмежень у картці моделі — так, як це зроблено в розділі «Limitations and bias» картки bert-base-uncased. Повністю усунути упередження складно, бо вони закладені в мільйонах речень, на яких модель навчалася, тому важливо їх вимірювати й зменшувати.

Мій додатковий експеримент підтверджує упередження: «The man was very [MASK].» → handsome, old, young; «The woman was very [MASK].» → pretty, beautiful, attractive — для жінки модель пропонує лише оцінки зовнішності.

### 5. Для чого в секції "Токенізація пари речень" виводяться числа 0 і 1?

Числа 0 і 1 — це `token_type_ids`, вони показують, до якого з двох речень належить кожен токен. BERT отримує обидва речення як одну послідовність `[CLS] речення 1 [SEP] речення 2 [SEP]`, і саме лише токен `[SEP]` не дуже надійно розділяє їх. Тому до ембедингу кожного токена додається ще ембединг сегмента: 0 — для `[CLS]`, першого речення і першого `[SEP]`, 1 — для другого речення і завершального `[SEP]`. У виводі видно: перші 20 токенів (`[CLS] revenue in the first quarter …`) мають мітку 0, останні 20 (`… earlier . [SEP]` другого речення) — мітку 1. Це потрібно моделі для задач на парах речень — під час попереднього тренування (Next Sentence Prediction) і для MRPC, де треба порівняти два речення.

### 6. Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?

Вивід `Linear(in_features=768, out_features=2, bias=True)` — це нова голова-класифікатор. 768 — розмір прихованого стану BERT-base (hidden size): на вхід голови подається вектор `[CLS]` (після шару pooler) довжиною 768 чисел. 2 це кількість виходів, тобто класів, які ми задали `num_labels=2`: «не парафраз» (0) і «парафраз» (1); голова видає два логіти, з яких softmax робить імовірності. Параметрів у голові 768 × 2 ваг + 2 зсуви (bias) = 1 538, що й показує наступна клітинка.

### 7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?

Для цієї задачі це добре. Кістяк (109 млн параметрів) уже під час попереднього тренування навчився розуміти мову: синтаксис, значення слів, звʼязки між реченнями. Голова (1 538 параметрів) — це лише лінійний шар, який має навчитися «читати» з вектора [CLS] відповідь так/ні. Для MRPC потрібне саме глибоке розуміння змісту двох речень, і всю цю роботу виконує кістяк, а голові досить мало параметрів, щоб розділити два класи. Мала голова також добре підходить для малого датасету (3 668 пар): велика голова з випадковими вагами легко перенавчилася б. Мінус, що під час повного донавчання оновлюються всі 110 млн параметрів, тому треба малий learning rate (2e-5) і небагато епох, щоб не «зламати» знання кістяка; через це ж тренування потребує більше памʼяті й часу, ніж навчання лише голови. Якби даних було зовсім мало, можна було б заморозити кістяк і тренувати тільки голову.

### 8. Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?

Після 3 епох на валідаційному наборі (408 пар): accuracy = 0.860, F1 = 0.902.

Accuracy — частка пар, для яких модель правильно визначила клас (парафраз чи ні): 86% з 408. F1 — гармонійне середнє precision (яка частка пар, названих моделлю парафразами, справді є парафразами) і recall (яку частку справжніх парафраз модель знайшла) для класу 1 «парафраз». F1 тут вища за accuracy, бо MRPC незбалансований: близько 68% пар — парафрази. Через це accuracy сама по собі може вводити в оману (модель, що завжди відповідає «парафраз», мала б ~68% accuracy і F1 ≈ 0.81), тому для MRPC стандартно наводять обидві метрики.

### 9. Збільшіть кількість епох тренування в 01_bert_head, покажіть (скопіювавши результати) що відбувається і чому.

Я донавчила ту саму модель на **8 епох** замість 3 (з тими самими гіперпараметрами) і записувала метрики після кожної епохи:

| епоха | train loss | val loss | val accuracy | val F1 |
|---|---|---|---|---|
| 1 | 0.529 | 0.377 | 0.836 | 0.881 |
| 2 | 0.306 | **0.354** | 0.851 | 0.896 |
| 3 | 0.147 | 0.525 | 0.855 | 0.891 |
| 4 | 0.071 | 0.643 | 0.863 | 0.903 |
| 5 | 0.042 | 0.713 | 0.860 | 0.903 |
| 6 | 0.016 | 0.816 | 0.863 | 0.904 |
| 7 | 0.017 | 0.805 | 0.865 | 0.906 |
| 8 | 0.010 | 0.784 | 0.863 | 0.903 |

для порівняння, 3 епохи в основній частині: val accuracy 0.860, F1 0.902.

train loss падає майже до нуля — модель фактично запамʼятовує 3 668 тренувальних пар. Validation loss мінімальний на 2-й епосі (0.354), а далі росте більш ніж удвічі (до 0.78–0.82) — перенавчання. При цьому accuracy і F1 майже не змінюються (0.86 / 0.90 ± 0.01): модель частіше не помиляється більше, але стає надмірно впевненою, і на тих прикладах, де вона помиляється, дає майже 100% упевненості в неправильній відповіді — за це крос-ентропія сильно штрафує, тому val loss росте. Це видно і в питанні 10: після 8 епох імовірності стали 100.0% і 0.2% замість 98% і 39%. Причина — 110 млн параметрів на малому датасеті: після 2–3 епох корисні закономірності вже вивчено, і подальше навчання підганяє модель під конкретні тренувальні приклади. Тому для MRPC вистачає 2–3 епох, а краще використовувати рання зупинку (`load_best_model_at_end` за validation loss/F1).

### 10. Додайте ще 4 перевірки на власних даних в пункті 4 в 01_bert_head.

Додала 4 власні пари й порівняла модель після 3 і після 8 епох:

| речення 1 | речення 2 | правильно | 3 епохи | 8 епох |
|---|---|---|---|---|
| He was late because the train was delayed. | The train delay made him late. | парафраз | 98.2% + | 100.0% + |
| The meeting starts at 10 am. | The meeting ends at 10 am. | **не** парафраз | 98.6% - | 100.0% - |
| I love this movie. | I do not like this movie at all. | не парафраз | 39.1% + | 0.2% + |
| The bank approved my loan application. | My loan request was accepted by the bank. | парафраз | 98.4% + | 100.0% + |

Модель правильно розпізнала перефразування з іншими словами («approved my loan application» ↔ «loan request was accepted») і протилежні за змістом речення про фільм. Але пару «starts at 10» / «ends at 10» вона вважає парафразом майже на 100%: речення відрізняються одним словом, і модель дивиться на загальний збіг слів, а не на зміну змісту — у MRPC мало таких «майже однакових, але протилежних» прикладів. Модель після 8 епох дає ті самі відповіді, але з упевненістю, близькою до 0 або 100% — це наслідок перенавчання з питання 9.